In [2]:
# 02_chunking.ipynb
# -------------------------------------------------------
# Flatten master.jsonl → chunk → save chunks.jsonl
# -------------------------------------------------------

import json
from pathlib import Path

SRC = Path("data/master.jsonl")
OUT = Path("data/chunks.jsonl")

CHUNK_SIZE = 1400
CHUNK_OVERLAP = 150

def extract_lang(val, lang):
    if isinstance(val, dict):
        return str(val.get(lang, ""))
    if isinstance(val, list):
        return ", ".join([extract_lang(x, lang) for x in val])
    return str(val)

def text_from_fields(obj, lang="en"):
    sections = []
    field_order = [
        "description","symptoms","causes","samprapti",
        "treatments","diet_and_lifestyle","prevention","conclusion"
    ]
    for key in field_order:
        val = obj.get(key)
        if not val:
            continue
        if isinstance(val, (dict, list)):
            sections.append(str(extract_lang(val, lang)))
        else:
            sections.append(str(val))
    return "\n".join([s for s in sections if str(s).strip()])

def simple_chunk(text, max_chars=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    if not words: return []
    chunks = []
    start = 0
    while start < len(words):
        end = start + max_chars
        chunks.append(" ".join(words[start:end]))
        start = end - overlap
        if start < 0: start = 0
    return chunks

with OUT.open("w", encoding="utf-8") as out:
    for line in SRC.open(encoding="utf-8"):
        obj = json.loads(line)
        disease_name_en = obj.get("disease", {}).get("en")
        disease_name_si = obj.get("disease", {}).get("si")
        for lang in ["en","si"]:
            text = text_from_fields(obj, lang)
            if not text.strip(): continue
            for i, chunk in enumerate(simple_chunk(text)):
                rec = {
                    "doc_id": obj["disease_id"],
                    "disease_name_en": disease_name_en,
                    "disease_name_si": disease_name_si,
                    "lang": lang,
                    "text": chunk,
                    "source": obj.get("metadata", {}).get("origin", ""),
                    "fields_present": list(obj.keys())
                }
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("Saved chunks.jsonl")


Saved chunks.jsonl
